# Pseudo-Log-Likelihood (PLL) Bias Analysis — PersianLLaMA (Persian)

This notebook computes PLL-based bias scores for **PersianLLaMA** on the Persian-translated StereoSet dataset.

**Metrics computed:**
- PLL per sentence (stereotype, anti-stereotype, unrelated) in Persian
- BiasScore = PLL(stereotype) − PLL(anti-stereotype)
- Categorical Bias (CB) per bias type

**Bias categories:** gender, race, religion, profession

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import math
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
# Load dataset
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)
df.head()

In [ ]:
# Load PersianLLaMA model
MODEL_NAME = 'Narabzad/PersianLLaMA-13B'
tokenizer_fa = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fa = AutoModelForCausalLM.from_pretrained(
 MODEL_NAME,
 torch_dtype=torch.float16,
 device_map='auto'
)
model_fa.eval()
print('PersianLLaMA loaded.')

In [ ]:
def compute_pll(sentence, tokenizer, model, device='cuda'):
 """
 Compute Pseudo-Log-Likelihood (PLL) for a sentence.
 """
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 input_ids = inputs['input_ids']
 with torch.no_grad():
 outputs = model(**inputs, labels=input_ids)
 n_tokens = input_ids.shape[1] - 1
 pll = -outputs.loss.item() * n_tokens
 return pll

In [ ]:
# Compute PLL for all Persian sentences
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pll_persian = []
for _, row in tqdm(df.iterrows(), total=len(df)):
 pll = compute_pll(row['Persian'], tokenizer_fa, model_fa, device)
 pll_persian.append(pll)

df['PLLPersian'] = pll_persian
df.head()

In [ ]:
# Compute BiasScore for Persian
bias_scores = []
for context, group in df.groupby('context'):
 stereo = group[group['label_name'] == 'stereotype']['PLLPersian'].values
 anti = group[group['label_name'] == 'anti-stereotype']['PLLPersian'].values
 if len(stereo) > 0 and len(anti) > 0:
 score = stereo[0] - anti[0]
 else:
 score = float('nan')
 for idx in group.index:
 if group.loc[idx, 'label_name'] != 'unrelated':
 bias_scores.append((idx, score))
 else:
 bias_scores.append((idx, float('nan')))

bias_df = pd.DataFrame(bias_scores, columns=['idx', 'BiasPersian']).set_index('idx')
df['BiasPersian'] = bias_df['BiasPersian']

In [ ]:
# Compute CB for Persian by bias type
print('--- Categorical Bias by Bias Type (PersianLLaMA) ---')
for btype, group in df.groupby('bias_type'):
 pll_stereo = group[group['label_name']=='stereotype']['PLLPersian'].mean()
 pll_anti = group[group['label_name']=='anti-stereotype']['PLLPersian'].mean()
 pll_unrel = group[group['label_name']=='unrelated']['PLLPersian'].mean()
 denom = abs(pll_stereo) + abs(pll_anti) + abs(pll_unrel)
 cb = (abs(pll_stereo) / denom) * 100
 print(f'{btype}: CB = {cb:.2f}')

In [ ]:
df.to_csv('/content/drive/MyDrive/stereoset_persianLLama.csv', index=False)
print('Saved stereoset_persianLLama.csv')